# Lab 17: Retrieval evaluation and the isolation test

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Fall-2026 | LLM mode: replay


In [2]:
import numpy as np, shutil
from src import sample_lsa
recs = sd.records("deal_a") + sd.records("deal_b")
chunks = [{"chunk_id": f'{r["doc_id"]}:{r["block_id"]}', "deal_id": r["deal_id"], "doc_id": r["doc_id"], "tier": r["tier"],
           "block_ids": [r["block_id"]], "text": r["text"]} for r in recs]
vecs = llm.embed([c["text"] for c in chunks], sample_lsa.MODEL, compute=sample_lsa.encode).tolist()
embed = lambda text: llm.embed([text], sample_lsa.MODEL, compute=sample_lsa.encode)[0].tolist()
import chromadb                    # rerun-safe: drop the collection through the client, not the folder
try:
    chromadb.PersistentClient(path="data/index").delete_collection("lantern")
except Exception:
    pass                           # first run: nothing to delete

In [3]:
# From the slide: A vector store with scope enforced in the store
import chromadb, json

client = chromadb.PersistentClient(path="data/index")
col = client.get_or_create_collection(
    "lantern", configuration={"hnsw": {"space": "cosine"}})

col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[{"deal_id": c["deal_id"],
                    "doc_id": c["doc_id"],
                    "tier": c["tier"],
                    "block_ids": json.dumps(c["block_ids"])}
                   for c in chunks])

def search(qvec, scope: str, k: int = 10):
    if not scope:
        raise ValueError("scope is required")
    return col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

In [4]:
# From the slide: BM25 plus dense, fused with reciprocal rank fusion
from rank_bm25 import BM25Okapi
tok = lambda s: s.lower().split()
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def rrf(*rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = (score.get(doc, 0)
                          + 1 / (k + rank))
    return sorted(score, key=score.get, reverse=True)

def hybrid(query, qvec, scope, k=10):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query))
    ok = (scope, "public")         # scope BM25 too
    lex = [chunks[i]["chunk_id"]
           for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex)[:k]

In [5]:
# From the slide: The metrics, implemented and tested
import math

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)

def mrr(ranked, rel):
    for i, d in enumerate(ranked, 1):
        if d in rel:
            return 1 / i
    return 0.0

def ndcg_at_k(ranked, grades, k):
    gain = lambda g, i: (2**g - 1) / math.log2(i + 1)
    dcg = sum(gain(grades.get(d, 0), i)
              for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(gain(g, i)
               for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

In [6]:
import pandas as pd
qrels = [json.loads(l) for l in open("data/eval/qrels.jsonl")]
rows = []
for q in qrels:
    rel, grades = list(q["relevant"]), q["relevant"]
    dense = search(embed(q["query"]), q["scope"], k=20)["ids"][0]
    hyb = hybrid(q["query"], embed(q["query"]), q["scope"], k=20)
    for name, ranked in (("dense", dense), ("hybrid", hyb)):
        rows.append({"retriever": name, "category": q["category"], "recall@5": recall_at_k(ranked, rel, 5),
                     "mrr": mrr(ranked, rel), "ndcg@10": ndcg_at_k(ranked, grades, 10)})
pd.DataFrame(rows).groupby("retriever")[["recall@5", "mrr", "ndcg@10"]].mean().round(3)

,recall@5,mrr,ndcg@10
retriever,,,
dense,1.000,1.000,0.958
hybrid,0.917,0.903,0.890


The isolation test from the slide runs unchanged: in this repository `data/eval/qrels.jsonl` and `data/dataroom/canaries_visible.json` are where Case Study 2 puts them.

In [10]:
chunk_meta = {c["chunk_id"]: c for c in chunks}.get

In [11]:
# From the slide: The isolation test, and proving it has teeth
import json, os, pytest
ROOT = os.environ.get("LANTERN_ISOLATION_FIXTURE", "data")
QRELS = f"{ROOT}/eval/qrels.jsonl"
CANS = f"{ROOT}/dataroom/canaries_visible.json"
QUERIES = [json.loads(l) for l in open(QRELS)]
CANARIES_B = json.load(open(CANS))["deal_b"]

@pytest.mark.parametrize("q", QUERIES)
def test_deal_a_never_sees_deal_b(q):
    hits = hybrid(q["query"], embed(q["query"]),
                  scope="deal_a", k=20)
    for cid in hits:
        meta = chunk_meta(cid)
        assert meta["deal_id"] in ("deal_a", "public")
        text = meta["text"]
        assert not any(c in text for c in CANARIES_B)

In [12]:
for q in QUERIES: test_deal_a_never_sees_deal_b(q)
print("isolation test passes on", len(QUERIES), "queries")
# Teeth: remove the scope filter and watch it fail
_safe = hybrid
def hybrid(query, qvec, scope, k=10):
    return [c["chunk_id"] for c in chunks][:k] + [c["chunk_id"] for c in chunks if c["deal_id"] == "deal_b"]
try:
    for q in QUERIES: test_deal_a_never_sees_deal_b(q)
    print("NO FAILURE: the test has no teeth")
except AssertionError:
    print("with the filter removed, the test fails: it has teeth")
hybrid = _safe

isolation test passes on 12 queries
with the filter removed, the test fails: it has teeth


**Exercise 17.** Write three more qrels for the transcript before looking at results. Which retriever wins on `cross_document` queries?

In [13]:
new_qrels = [
    {
        "qid": "w-13",
        "query": (
            "How does the CFO's trailing net revenue retention "
            "explanation compare with the CIM's 118% claim?"
        ),
        "category": "cross_document",
        "scope": "deal_a",
        "relevant": {
            "DR-deal_a-002:t003": 2,
            "DR-deal_a-001:p0003_b002": 2,
        },
    },
    {
        "qid": "w-14",
        "query": (
            "How does the CEO describe customer concentration, "
            "and what numerical limit does the CIM claim?"
        ),
        "category": "cross_document",
        "scope": "deal_a",
        "relevant": {
            "DR-deal_a-002:t002": 2,
            "DR-deal_a-001:p0003_b001": 2,
        },
    },
    {
        "qid": "w-15",
        "query": (
            "Compare management's comments about change-of-control "
            "consent with the termination right in the Northline contract."
        ),
        "category": "cross_document",
        "scope": "deal_a",
        "relevant": {
            "DR-deal_a-002:t004": 2,
            "DR-deal_a-003:p0001_b003": 2,
        },
    },
]

# Save the labels before examining retrieval results.
exercise_path = Path("data/eval/qrels_exercise17.jsonl")
exercise_path.write_text(
    "\n".join(json.dumps(q) for q in new_qrels) + "\n"
)

print("Saved three new labelled questions.")

Saved three new labelled questions.


In [14]:
# Cache embeddings for the new questions locally.
previous_mode = llm.MODE
try:
    llm.MODE = "record"
    llm.embed(
        [q["query"] for q in new_qrels],
        sample_lsa.MODEL,
        compute=sample_lsa.encode,
    )
finally:
    llm.MODE = previous_mode

original_qrels = [
    json.loads(line)
    for line in Path("data/eval/qrels.jsonl").read_text().splitlines()
    if line.strip()
]

# Merge by ID so rerunning does not duplicate questions.
all_qrels = {
    q["qid"]: q for q in original_qrels + new_qrels
}.values()

results = []

for q in all_qrels:
    if q["category"] != "cross_document":
        continue

    qvec = embed(q["query"])
    rankings = {
        "vector": search(qvec, q["scope"], k=20)["ids"][0],
        "hybrid": hybrid(q["query"], qvec, q["scope"], k=20),
    }

    for name, ranked in rankings.items():
        results.append({
            "qid": q["qid"],
            "retriever": name,
            "recall@5": recall_at_k(ranked, q["relevant"], 5),
            "mrr": mrr(ranked, q["relevant"]),
            "ndcg@10": ndcg_at_k(ranked, q["relevant"], 10),
        })

details = pd.DataFrame(results)
summary = details.groupby("retriever")[
    ["recall@5", "mrr", "ndcg@10"]
].mean()

display(details)
display(summary.round(3))

,qid,retriever,recall@5,mrr,ndcg@10
0,w-06,vector,1.0,1.0,1.000000
1,w-06,hybrid,0.5,1.0,0.797723
2,w-12,vector,1.0,1.0,0.932778
3,w-12,hybrid,1.0,1.0,0.944848
4,w-13,vector,1.0,1.0,1.000000
5,w-13,hybrid,1.0,1.0,1.000000
6,w-14,vector,1.0,1.0,0.919721
7,w-14,hybrid,1.0,1.0,0.877215
8,w-15,vector,1.0,1.0,1.000000
9,w-15,hybrid,0.5,1.0,0.806574


,recall@5,mrr,ndcg@10
retriever,,,
hybrid,0.8,1.0,0.885
vector,1.0,1.0,0.970
